In [ ]:
import pandas as pd
from datasets import load_dataset

data = load_dataset("sh0416/ag_news")

(
    pd.Series(data["train"]["description"])  # type: ignore
    .str.len()
    .plot.hist(
        bins=20,
        xlabel="Text length",
    )
);

In [ ]:
(
    pd.Series(data["train"]["label"])  # type: ignore
    .value_counts()
    .sort_index()
    .plot.bar(
        xlabel="Label",
        ylabel="Frequency",
        rot=0,
    )
);

In [ ]:
import os
import json
import time
import torch

from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("google/bert-base-ucased")
model = AutoModelForSequenceClassification.from_pretrained("./bert-ag/checkpoint-816")
model.to(device)
model.eval()

print(f"Model Size: {sum(p.numel() for p in model.parameters()):,} parameters")


dataset = load_dataset("jziebura/polish_youth_slang_classification")
dataset = dataset.rename_column("sentyment", "labels")


def tokenize(examples):
    return tokenizer(examples["tekst"], padding="max_length", truncation=True, max_length=512)


# Only process the test set
test_dataset = dataset["test"].map(tokenize, batched=True)  # type: ignore
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])  # type: ignore
test_loader = DataLoader(test_dataset, batch_size=16)  # type: ignore


# 5. INFERENCE TIME & ACCURACY
print("Running Inference...")
all_preds = []
all_labels = []

t = time.time()

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["labels"]

        outputs = model(input_ids, attention_mask=mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()

        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

t = time.time() - t

# REPORT
print("-" * 30)
print("FINAL RESULTS")
print("-" * 30)

# Metrics
print(f"Accuracy:            {accuracy_score(all_labels, all_preds):.4f}")
print(f"F1 Score (Weighted): {f1_score(all_labels, all_preds, average="weighted"):.4f}")
print(f"F1 Score (Macro):    {f1_score(all_labels, all_preds, average="macro"):.4f}")
print("-" * 30)

# Inference Speed
num_samples = len(all_labels)
print(f"Total Inference Time: {t:.2f} s")
print(f"Time per Sample:      {(t / num_samples) * 1000:.2f} ms")
print(f"Samples per Second:   {num_samples / t:.2f}")

In [ ]:
import matplotlib.pyplot as plt


def plot_training_curves(model_dir):
    with open(os.path.join(model_dir, "trainer_state.json"), "r") as f:
        data = json.load(f)

    history = data["log_history"]

    # 2. Extract Data
    train_steps, train_loss = [], []
    val_steps, val_loss, val_acc, val_f1 = [], [], [], []

    for entry in history:
        if "loss" in entry:
            train_steps.append(entry["step"])
            train_loss.append(entry["loss"])
        if "eval_loss" in entry:
            val_steps.append(entry["step"])
            val_loss.append(entry["eval_loss"])
            val_acc.append(entry["eval_accuracy"])
            val_f1.append(entry["eval_f1"])

    plt.figure(figsize=(15, 6))
    plt.subplot(1, 2, 1)
    plt.plot(train_steps, train_loss, label="Training Loss")
    plt.plot(val_steps, val_loss, label="Validation Loss", marker=".")
    plt.xlabel("Steps")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.plot(val_steps, val_acc, label="Accuracy", marker=".")
    plt.plot(val_steps, val_f1, label="F1 Score", marker=".")
    plt.xlabel("Steps")
    plt.ylabel("Score")
    plt.title("Validation Metrics")
    plt.legend()
    plt.ylim(0, 1.05)
    plt.show()


plot_training_curves("./herbert-slang/checkpoint-816")